# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
# Load environment variables
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

### VectorDB Instance

In [5]:
# Instantiate your ChromaDB Client
chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [6]:
# Pick one embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# Create a collection
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)
print(f"Collection '{collection.name}' is ready.")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection 'udaplay' is ready.


### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

if not os.path.exists(data_dir):
    print(f"Directory '{data_dir}' not found.")
else:
    added_count = 0

    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                game = json.load(f)
            except json.JSONDecodeError:
                print(f"Skipping malformed JSON file: {file_name}")
                continue
        
        # Format text to index
        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

        # Use file name (like 001) as ID
        doc_id = os.path.splitext(file_name)[0]

        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game]
        )
        added_count += 1

    print(f"Successfully processed and inserted {added_count} games into collection.")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Successfully processed and inserted 15 games into collection.


In [9]:
# Test semantic search
def semantic_search(query: str, n_results: int = 3) -> list:
    """Search collection and return matches with metadata and similarity scores."""
    results = collection.query(query_texts=[query], n_results=n_results)
    matches = []
    for document, metadata, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        matches.append({**metadata, "similarity": round(1 - distance, 3), "text": document})
    return matches


def print_matches(query: str, n_results: int = 3):
    print(f"Query: {query!r}")
    for m in semantic_search(query, n_results):
        print(
            f"[Similarity Score: {m['similarity']:.3f}] "
            f"{m['Name']} ({m['YearOfRelease']}, {m['Platform']}) - {m['Publisher']}"
        )
    print()

In [10]:
print_matches("When was Pokémon Gold and Silver released?")

Query: 'When was Pokémon Gold and Silver released?'


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Similarity Score: 0.774] Pokémon Gold and Silver (1999, Game Boy Color) - Nintendo
[Similarity Score: 0.707] Pokémon Ruby and Sapphire (2002, Game Boy Advance) - Nintendo
[Similarity Score: 0.559] Super Mario 64 (1996, Nintendo 64) - Nintendo



In [11]:
print_matches("Which one was the first 3D platformer Mario game?")

Query: 'Which one was the first 3D platformer Mario game?'
[Similarity Score: 0.770] Super Mario 64 (1996, Nintendo 64) - Nintendo
[Similarity Score: 0.718] Super Mario World (1990, Super Nintendo Entertainment System (SNES)) - Nintendo
[Similarity Score: 0.588] Mario Kart 8 Deluxe (2017, Nintendo Switch) - Nintendo



In [12]:
print_matches("Was Mortal Kombat X released for PlayStation 5?")

Query: 'Was Mortal Kombat X released for PlayStation 5?'
[Similarity Score: 0.597] Marvel's Spider-Man 2 (2023, PlayStation 5) - Sony Interactive Entertainment
[Similarity Score: 0.560] Marvel's Spider-Man (2018, PlayStation 4) - Sony Interactive Entertainment
[Similarity Score: 0.560] Halo Infinite (2021, Xbox Series X|S) - Xbox Game Studios

